# Task 3 - Model Comparison (Kaggle/Colab)
Load outputs from SARIMA/LSTM/GRU and build plots + tables.

In [ ]:
# Load model outputs
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Prefer Kaggle working output first, then local notebook output.
OUTPUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
if not OUTPUT_DIR.exists():
    fallback = Path("outputs")
    if fallback.exists():
        OUTPUT_DIR = fallback
print(f"Using outputs from: {OUTPUT_DIR.resolve()}")

def load_metrics(path: Path, model_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing metrics file: {path}")
    df = pd.read_csv(path)
    df["model"] = model_name
    return df

metrics = pd.concat([
    load_metrics(OUTPUT_DIR / "metrics_sarima.csv", "SARIMA"),
    load_metrics(OUTPUT_DIR / "metrics_lstm.csv", "LSTM"),
    load_metrics(OUTPUT_DIR / "metrics_gru.csv", "GRU"),
], ignore_index=True)
metrics

In [ ]:
# Metrics tables by square
metrics_table = metrics.pivot_table(index=["square_id", "model"], values=["MAE", "MAPE", "RMSE"]).reset_index()
metrics_table

In [ ]:
# Forecast overlay plots (9 total)
def plot_overlay(csv_path: Path, title: str):
    if not csv_path.exists():
        print(f"Skipping missing forecast file: {csv_path}")
        return None
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pd.to_datetime(df["time_interval"], utc=True), df["y_true"], label="Observed", linewidth=1.8)
    ax.plot(pd.to_datetime(df["time_interval"], utc=True), df["y_pred"], label="Predicted", linewidth=1.6, alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Internet traffic")
    ax.legend()
    fig.tight_layout()
    return fig

for model_name in ["sarima", "lstm", "gru"]:
    for square_id in metrics["square_id"].unique():
        path = OUTPUT_DIR / f"{model_name}_{int(square_id)}.csv"
        if path.exists():
            plot_overlay(path, f"{model_name.upper()} - Square {int(square_id)} (Dec 16-22)")